In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

os.environ["TDF_DATASET_ROOT"] = (
    "/content/drive/.shortcut-targets-by-id/"
    "1TqpwmwVmWbDv2rn_DsNCh1aVjB1QZwca/"
    "PlantVillage/plantvillage dataset/color"
)

dataset_root = os.environ["TDF_DATASET_ROOT"]

print("Dataset path:", dataset_root)
print("Path exists:", os.path.isdir(dataset_root))

if os.path.isdir(dataset_root):
    print("Number of class folders:", len([
        name for name in os.listdir(dataset_root)
        if os.path.isdir(os.path.join(dataset_root, name))
    ]))
else:
    print("ERROR: Check that Drive is mounted and the path is correct.")

Dataset path: /content/drive/.shortcut-targets-by-id/1TqpwmwVmWbDv2rn_DsNCh1aVjB1QZwca/PlantVillage/plantvillage dataset/color
Path exists: True
Number of class folders: 10


In [5]:
%cd /content
!git clone https://github.com/parnikaaa06/tomato-disease-framework.git

/content
Cloning into 'tomato-disease-framework'...
remote: Enumerating objects: 222, done.
remote: Counting objects: 100% (222/222), done.
remote: Compressing objects: 100% (152/152), done.
remote: Total 222 (delta 76), reused 198 (delta 58), pack-reused 0 (from 0)
Receiving objects: 100% (222/222), 4.26 MiB | 16.99 MiB/s, done.
Resolving deltas: 100% (76/76), done.


In [6]:
%cd /content/tomato-disease-framework/tomato_disease_framework

/content/tomato-disease-framework/tomato_disease_framework


In [7]:
import os

print("Folders in /content:")
print(os.listdir("/content"))

Folders in /content:
['.config', 'drive', 'tomato-disease-framework', 'sample_data']


In [8]:
import os

print("Current folder:", os.getcwd())
print(
    "Phase 7 script exists:",
    os.path.isfile("scripts/audit_severity_progression.py")
)
print(
    "Phase 7 notebook exists:",
    os.path.isfile("notebooks/phase7_severity_progression_feasibility.ipynb")
)

Current folder: /content/tomato-disease-framework/tomato_disease_framework
Phase 7 script exists: True
Phase 7 notebook exists: True


In [9]:
import os

dataset_root = (
    "/content/drive/.shortcut-targets-by-id/"
    "1TqpwmwVmWbDv2rn_DsNCh1aVjB1QZwca/"
    "PlantVillage/plantvillage dataset/color"
)

os.environ["TDF_DATASET_ROOT"] = dataset_root

print("Dataset path exists:", os.path.isdir(dataset_root))

if os.path.isdir(dataset_root):
    print("Number of class folders:", len([
        name for name in os.listdir(dataset_root)
        if os.path.isdir(os.path.join(dataset_root, name))
    ]))

Dataset path exists: True
Number of class folders: 10


# Phase 7 — Severity and Progression Feasibility Audit

This is a read-only audit of whether the available PlantVillage data and project metadata contain defensible ground-truth evidence for disease severity, grades/stages, disease-area measurements, temporal progression, and repeated observations of the same plant or leaf. It does not train a model.

Disease class names, visual symptoms, similar filenames, and image similarity are not severity labels, area measurements, timestamps, or plant/leaf identity.

## 1. Project and dataset path

Mount Google Drive in Colab first if needed. Set `TDF_DATASET_ROOT` to the existing PlantVillage `color` directory; the audit does not alter raw images. If the dataset is not mounted, the script reports that limitation and audits the tracked repository metadata only.

In [10]:
from pathlib import Path
import os
import sys

repo_root = Path.cwd()
if not (repo_root / 'scripts' / 'audit_severity_progression.py').is_file():
    candidates = [parent for parent in repo_root.parents if (parent / 'scripts' / 'audit_severity_progression.py').is_file()]
    if candidates:
        repo_root = candidates[0]
if not (repo_root / 'scripts' / 'audit_severity_progression.py').is_file():
    raise FileNotFoundError('Run this notebook from the repository or clone the project first.')
os.chdir(repo_root)

if not os.environ.get('TDF_DATASET_ROOT'):
    os.environ['TDF_DATASET_ROOT'] = '/content/drive/MyDrive/UROP_Plant_Disease/datasets/PlantVillage/plantvillage dataset/color'
dataset_root = Path(os.environ['TDF_DATASET_ROOT']).expanduser()
print('Repository:', repo_root)
print('Dataset root:', dataset_root)
print('Dataset root available:', dataset_root.is_dir())

Repository: /content/tomato-disease-framework/tomato_disease_framework
Dataset root: /content/drive/.shortcut-targets-by-id/1TqpwmwVmWbDv2rn_DsNCh1aVjB1QZwca/PlantVillage/plantvillage dataset/color
Dataset root available: True


## 2. Environment verification

In [11]:
import yaml
import pandas as pd

print('Python:', sys.version.split()[0])
print('PyYAML:', yaml.__version__)
print('pandas:', pd.__version__)
print('Environment ready for audit; no model training dependencies are invoked.')

Python: 3.13.16
PyYAML: 6.0.3
pandas: 2.2.3
Environment ready for audit; no model training dependencies are invoked.


## 3. Run the read-only audit script

The script reads structured metadata, manifests, and split tables. Candidate field names are discovery leads only; a populated field does not establish annotation provenance or validity.

In [12]:
import subprocess

script_path = repo_root / 'scripts' / 'audit_severity_progression.py'
completed = subprocess.run(
    [sys.executable, str(script_path)],
    cwd=repo_root,
    env=os.environ.copy(),
    capture_output=True,
    text=True,
    check=False,
)
print(completed.stdout)
if completed.stderr:
    print(completed.stderr)
if completed.returncode != 0:
    raise RuntimeError(f'Audit script failed with exit code {completed.returncode}.')

Dataset root status: available
Structured metadata sources inspected: 6
Candidate inventory written to: /content/tomato-disease-framework/tomato_disease_framework/results/phase7/severity_progression_candidate_inventory.csv
Feasibility report written to: /content/tomato-disease-framework/tomato_disease_framework/results/phase7/severity_progression_feasibility_report.yaml
Defensible ground truth for severity, grade/stage, disease-area measurement, temporal progression, and repeated observations of the same plant or leaf has not been established from inspected sources. Severity/progression model training is not justified.
  severity_target: NOT ESTABLISHED
  severity_grades_or_stages: NOT ESTABLISHED
  disease_area_measurements: NOT ESTABLISHED
  temporal_progression: NOT ESTABLISHED
  repeated_plant_or_leaf_observations: NOT ESTABLISHED
  severity_or_progression_model_training: NOT JUSTIFIED



## 4. Candidate metadata inventory

In [13]:
import json

output_dir = repo_root / 'results' / 'phase7'
inventory_path = output_dir / 'severity_progression_candidate_inventory.csv'
report_path = output_dir / 'severity_progression_feasibility_report.yaml'
inventory = pd.read_csv(inventory_path)
with report_path.open('r', encoding='utf-8') as handle:
    report = yaml.safe_load(handle)

print('Dataset root status:', report['dataset_root_status'])
print('Dataset file scan:', report['dataset_file_scan'])
print('Metadata sources inventoried:', report['metadata_inventory_count'])
display(inventory if not inventory.empty else pd.DataFrame({'status': ['No structured candidate metadata sources found.']}))

Dataset root status: available
Dataset file scan: {'status': 'completed', 'image_count': 18160, 'class_counts': {'Tomato___Bacterial_spot': 2127, 'Tomato___Early_blight': 1000, 'Tomato___Late_blight': 1909, 'Tomato___Leaf_Mold': 952, 'Tomato___Septoria_leaf_spot': 1771, 'Tomato___Spider_mites Two-spotted_spider_mite': 1676, 'Tomato___Target_Spot': 1404, 'Tomato___Tomato_Yellow_Leaf_Curl_Virus': 5357, 'Tomato___Tomato_mosaic_virus': 373, 'Tomato___healthy': 1591}}
Metadata sources inventoried: 6


,source_path,file_type,status,record_count,fields,candidate_field_groups,records_with_populated_candidate_fields,repeated_observation_check,error
0,/content/tomato-disease-framework/tomato_disea...,.json,inspected,1,classification_labels_are_not_masks_or_severit...,{},{},"{""identity_fields"": [], ""identity_groups_with_...",NaN
1,/content/tomato-disease-framework/tomato_disea...,.json,inspected,1,class_count; class_counts.Tomato___Bacterial_s...,{},{},"{""identity_fields"": [], ""identity_groups_with_...",NaN
2,/content/tomato-disease-framework/tomato_disea...,.csv,inspected,18160,relative_file_path; class_name; file_name; ext...,{},{},"{""identity_fields"": [], ""identity_groups_with_...",NaN
3,/content/tomato-disease-framework/tomato_disea...,.csv,inspected,2725,image_path; class_name,{},{},"{""identity_fields"": [], ""identity_groups_with_...",NaN
4,/content/tomato-disease-framework/tomato_disea...,.csv,inspected,12710,image_path; class_name,{},{},"{""identity_fields"": [], ""identity_groups_with_...",NaN
5,/content/tomato-disease-framework/tomato_disea...,.csv,inspected,2725,image_path; class_name,{},{},"{""identity_fields"": [], ""identity_groups_with_...",NaN


## 5. Evidence-based target assessment

Only explicit severity/grade, disease-area, plant/leaf identity, and time fields are considered as possible metadata leads. The script does not infer labels from class names or estimate area from pixels.

In [14]:
display(pd.DataFrame([
    {'Question': question.replace('_', ' ').capitalize(), 'Status': status}
    for question, status in report['questions'].items()
]))
print('Candidate field evidence:')
display(pd.DataFrame([
    {'Field group': group.replace('_', ' ').capitalize(), **evidence}
    for group, evidence in report['candidate_field_evidence'].items()
]) if report['candidate_field_evidence'] else pd.DataFrame({'Evidence': ['No populated candidate target fields were found in inspected metadata.']}))

,Question,Status
0,Disease severity ground truth,NOT ESTABLISHED
1,Severity grades or stages,NOT ESTABLISHED
2,Disease area measurements,NOT ESTABLISHED
3,Temporal progression,NOT ESTABLISHED
4,Repeated same plant or leaf observations,NOT ESTABLISHED


Candidate field evidence:


,Evidence
0,No populated candidate target fields were foun...


## 6. Repeated-observation and progression checks

Repeated observations require an explicit plant/leaf identifier and distinct observation dates in the same metadata source. Duplicate images, filename similarity, and disease categories are not evidence of a longitudinal series.

In [15]:
repeat_checks = []
for source in report['metadata_inventory']:
    check = source['repeated_observation_check']
    repeat_checks.append({
        'Source': source['source_path'],
        'Explicit identity fields': ', '.join(check['identity_fields']),
        'Explicit time fields': ', '.join(check['time_fields']),
        'Records with identity and date': check['records_with_identity_and_time'],
        'Identity groups with multiple dates': check['identity_groups_with_multiple_dates'],
        'Repeated observation evidence': check['repeated_observation_evidence'],
    })
display(pd.DataFrame(repeat_checks) if repeat_checks else pd.DataFrame({'Status': ['No structured sources available for an identity/date check.']}))
print('Candidate repeat-observation source files:', report['repeated_observation_candidate_sources'])

,Source,Explicit identity fields,Explicit time fields,Records with identity and date,Identity groups with multiple dates,Repeated observation evidence
0,/content/tomato-disease-framework/tomato_disea...,,,0,0,False
1,/content/tomato-disease-framework/tomato_disea...,,,0,0,False
2,/content/tomato-disease-framework/tomato_disea...,,,0,0,False
3,/content/tomato-disease-framework/tomato_disea...,,,0,0,False
4,/content/tomato-disease-framework/tomato_disea...,,,0,0,False
5,/content/tomato-disease-framework/tomato_disea...,,,0,0,False


Candidate repeat-observation source files: []


## 7. Phase 7 decision

A metadata field is not validated ground truth until its source, collection protocol, units/definitions, and linkage to the image are verified. This audit is a feasibility gate only; it does not fabricate severity labels, infer temporal progression, or train a severity/progression model.

In [16]:
print(report['summary'])
display(pd.DataFrame([
    {'Capability': name.replace('_', ' ').capitalize(), 'Decision': status}
    for name, status in report['conclusions'].items()
]))
print('Manual review required:', report['manual_review_required'])
for limitation in report['evidence_and_limitations']:
    print('-', limitation)

Defensible ground truth for severity, grade/stage, disease-area measurement, temporal progression, and repeated observations of the same plant or leaf has not been established from inspected sources. Severity/progression model training is not justified.


,Capability,Decision
0,Severity target,NOT ESTABLISHED
1,Severity grades or stages,NOT ESTABLISHED
2,Disease area measurements,NOT ESTABLISHED
3,Temporal progression,NOT ESTABLISHED
4,Repeated plant or leaf observations,NOT ESTABLISHED
5,Severity or progression model training,NOT JUSTIFIED


Manual review required: True
- Plant disease class labels identify categories, not disease severity, severity grade, or disease stage.
- Image appearance and lesion-like pixels were not converted into severity scores or affected-area measurements.
- Filename similarity and duplicate-image records were not treated as repeated observations of the same plant or leaf.
- A field name or populated metadata value is only a discovery lead; annotation provenance, definitions, and protocol must be validated before establishing ground truth.
- The audit does not train a severity model or infer temporal progression.


In [17]:
from pathlib import Path

results_dir = Path("results/phase7")

for filename in [
    "severity_progression_feasibility_report.yaml",
    "severity_progression_candidate_inventory.csv",
]:
    path = results_dir / filename
    print(f"\n{filename}")
    print("Exists:", path.exists())

    if path.exists():
        print("Size:", path.stat().st_size, "bytes")
        if path.suffix == ".yaml":
            print(path.read_text())


severity_progression_feasibility_report.yaml
Exists: True
Size: 6717 bytes
dataset_root: /content/drive/.shortcut-targets-by-id/1TqpwmwVmWbDv2rn_DsNCh1aVjB1QZwca/PlantVillage/plantvillage
  dataset/color
dataset_root_status: available
dataset_file_scan:
  status: completed
  image_count: 18160
  class_counts:
    Tomato___Bacterial_spot: 2127
    Tomato___Early_blight: 1000
    Tomato___Late_blight: 1909
    Tomato___Leaf_Mold: 952
    Tomato___Septoria_leaf_spot: 1771
    Tomato___Spider_mites Two-spotted_spider_mite: 1676
    Tomato___Target_Spot: 1404
    Tomato___Tomato_Yellow_Leaf_Curl_Virus: 5357
    Tomato___Tomato_mosaic_virus: 373
    Tomato___healthy: 1591
inspected_scope:
- configured image root and selected sibling metadata directories
- repository metadata/split artifacts
metadata_inventory_count: 6
metadata_inventory:
- source_path: /content/tomato-disease-framework/tomato_disease_framework/data/dataset_report/annotation_availability.json
  file_type: .json
  status: ins

In [18]:
import pandas as pd

csv_path = Path(
    "results/phase7/severity_progression_candidate_inventory.csv"
)

if csv_path.exists():
    df = pd.read_csv(csv_path)
    print("Rows:", len(df))
    display(df.head())
    print("Columns:", df.columns.tolist())

Rows: 6


,source_path,file_type,status,record_count,fields,candidate_field_groups,records_with_populated_candidate_fields,repeated_observation_check,error
0,/content/tomato-disease-framework/tomato_disea...,.json,inspected,1,classification_labels_are_not_masks_or_severit...,{},{},"{""identity_fields"": [], ""identity_groups_with_...",NaN
1,/content/tomato-disease-framework/tomato_disea...,.json,inspected,1,class_count; class_counts.Tomato___Bacterial_s...,{},{},"{""identity_fields"": [], ""identity_groups_with_...",NaN
2,/content/tomato-disease-framework/tomato_disea...,.csv,inspected,18160,relative_file_path; class_name; file_name; ext...,{},{},"{""identity_fields"": [], ""identity_groups_with_...",NaN
3,/content/tomato-disease-framework/tomato_disea...,.csv,inspected,2725,image_path; class_name,{},{},"{""identity_fields"": [], ""identity_groups_with_...",NaN
4,/content/tomato-disease-framework/tomato_disea...,.csv,inspected,12710,image_path; class_name,{},{},"{""identity_fields"": [], ""identity_groups_with_...",NaN


Columns: ['source_path', 'file_type', 'status', 'record_count', 'fields', 'candidate_field_groups', 'records_with_populated_candidate_fields', 'repeated_observation_check', 'error']


In [19]:
from pathlib import Path
import shutil

# Source: generated Phase 7 reports in your Colab project
source = Path(
    "/content/tomato-disease-framework/"
    "tomato_disease_framework/results/phase7"
)

# Destination: your persistent Google Drive
destination = Path(
    "/content/drive/MyDrive/UROP_Plant_Disease/results/phase7"
)

files_to_copy = [
    "severity_progression_candidate_inventory.csv",
    "severity_progression_feasibility_report.yaml",
]

destination.mkdir(parents=True, exist_ok=True)

for filename in files_to_copy:
    source_file = source / filename

    if not source_file.exists():
        raise FileNotFoundError(f"Report not found: {source_file}")

    shutil.copy2(source_file, destination / filename)
    print(f"Copied: {filename}")

print("\nDestination files:")
for path in sorted(destination.iterdir()):
    if path.is_file():
        print(path.name, "-", path.stat().st_size, "bytes")

Copied: severity_progression_candidate_inventory.csv
Copied: severity_progression_feasibility_report.yaml

Destination files:
severity_progression_candidate_inventory.csv - 3386 bytes
severity_progression_feasibility_report.yaml - 6717 bytes
